In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into kernel variables (full DataFrames kept for later steps)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Convert timestamp columns to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to build compact overview and top-N counts for a field
def compact_summary(df, id_col, name_col, top_n=20):
    total_rows = len(df)
    unique_cmdb = sorted(df[id_col].dropna().unique()) if id_col in df.columns else []
    unique_cmdb_count = len(unique_cmdb)
    # build compact string of unique cmdb ids (limit to 20 in display)
    if unique_cmdb_count > top_n:
        unique_cmdb_display = ','.join(unique_cmdb[:top_n]) + ',...'
    else:
        unique_cmdb_display = ','.join(unique_cmdb)
    # time range
    if 'ts' in df.columns and not df['ts'].isna().all():
        min_ts = df['ts'].min()
        max_ts = df['ts'].max()
    else:
        min_ts = pd.NaT
        max_ts = pd.NaT
    # top counts for name_col
    if name_col in df.columns:
        top_counts = (
            df.groupby(name_col)
              .size()
              .reset_index(name='count')
              .sort_values('count', ascending=False)
              .head(top_n)
              .reset_index(drop=True)
        )
    else:
        top_counts = pd.DataFrame(columns=[name_col, 'count'])
    overview = pd.DataFrame([{
        'total_rows': total_rows,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_values': unique_cmdb_display,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts
    }])
    return overview, top_counts

# Produce summaries for each file
metric_overview, metric_topk = compact_summary(df_metric, 'cmdb_id', 'kpi_name', top_n=20)
trace_overview, trace_topk = compact_summary(df_trace, 'cmdb_id', 'trace_name', top_n=20)
log_overview, log_topk = compact_summary(df_log, 'cmdb_id', 'log_name', top_n=20)
# For error_logs, group by 'message' (or show samples if mostly unique)
error_overview, error_topk = compact_summary(df_error, 'cmdb_id', 'message', top_n=20)

# Keep final displayed output compact: show overviews and top-20 groups
metric_overview, metric_topk, trace_overview, trace_topk, log_overview, log_topk, error_overview, error_topk

```
Out[1]:
```
# Summarize the previously computed compact aggregates in plain English.
# Use existing kernel variables: metric_overview, metric_topk, trace_overview, trace_topk, log_overview, log_topk, error_overview, error_topk

def top_list_to_str(df, name_col, max_items=5):
    if df.empty:
        return "None"
    items = df.head(max_items).apply(lambda r: f"{r[name_col]} ({int(r['count'])})", axis=1).tolist()
    more = ""
    if len(df) > max_items:
        more = f", ... (+{len(df)-max_items} more)"
    return ", ".join(items) + more

m = metric_overview.iloc[0]
t = trace_overview.iloc[0]
l = log_overview.iloc[0]
e = error_overview.iloc[0]

summary = (
    "Summary of loaded telemetry files:\n\n"
    "1) metric.csv:\n"
    f"   - Total rows: {int(m['total_rows'])}\n"
    f"   - Unique cmdb_id count: {int(m['unique_cmdb_count'])}\n"
    f"   - cmdb_id samples: {m['unique_cmdb_values']}\n"
    f"   - Time range: {m['min_timestamp']} to {m['max_timestamp']}\n"
    f"   - Top metric kpi_name (top 5 shown): {top_list_to_str(metric_topk, 'kpi_name', 5)}\n\n"
    "2) trace.csv:\n"
    f"   - Total rows: {int(t['total_rows'])}\n"
    f"   - Unique cmdb_id count: {int(t['unique_cmdb_count'])}\n"
    f"   - cmdb_id samples: {t['unique_cmdb_values']}\n"
    f"   - Time range: {t['min_timestamp']} to {t['max_timestamp']}\n"
    f"   - Top trace_name (top 5 shown): {top_list_to_str(trace_topk, 'trace_name', 5)}\n\n"
    "3) log.csv:\n"
    f"   - Total rows: {int(l['total_rows'])}\n"
    f"   - Unique cmdb_id count: {int(l['unique_cmdb_count'])}\n"
    f"   - cmdb_id samples: {l['unique_cmdb_values']}\n"
    f"   - Time range: {l['min_timestamp']} to {l['max_timestamp']}\n"
    f"   - Top log_name (top 5 shown): {top_list_to_str(log_topk, 'log_name', 5)}\n\n"
    "4) error_logs.csv:\n"
    f"   - Total rows: {int(e['total_rows'])}\n"
    f"   - Unique cmdb_id count: {int(e['unique_cmdb_count'])}\n"
    f"   - Time range: {e['min_timestamp']} to {e['max_timestamp']}\n"
    "   - No error messages present (empty file).\n\n"
    "Notes:\n"
    "- All timestamps are shown in UTC+8 (Asia/Shanghai).\n"
    "- Top lists show counts; full top-20 tables are available in kernel variables metric_topk, trace_topk, log_topk, error_topk."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  unique_cmdb_count                                 unique_cmdb_values             min_timestamp             max_timestamp
0       30676                 25  IG01,IG02,MG01,MG02,Mysql01,Mysql02,Redis01,Re... 2021-03-23 01:00:00+08:00 2021-03-23 01:30:00+08:00,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    365
1        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    364
2       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    361
3                         OSLinux-CPU_CPU_CPUidleutil    361
4          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    361
5                  OSLinux-OSLinux_ZABBIX_Host_Uptime    361
6                             OSLinux-CPU_CPU_CPULoad    361
7                          OSLinux-CPU_CPU_CPUSysTime    357
8                         OSLinux-CPU_CPU_CPUUserTime    357
9                              OSLinux-CPU_CPU_CPUWio    357
10  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    357
11                                             app.rr    330
12                                            app.cnt    330
13                                            app.mrt    330
14                                             app.sr    330
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    321
16       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    307
17                         OSLinux-CPU_CPU_CPUCpuUtil    306
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    304
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    295,    total_rows  unique_cmdb_count                                 unique_cmdb_values             min_timestamp             max_timestamp
0        6798                 12  IG01,IG02,MG01,MG02,Tomcat01,Tomcat02,Tomcat03... 2021-03-23 01:00:00+08:00 2021-03-23 01:30:00+08:00,                        trace_name  count
0         trace.self.duration_p95    365
1            trace.self.row_count    365
2        trace.self.duration_mean    365
3     trace.to_MG01.duration_mean    237
4         trace.to_MG01.row_count    237
5      trace.to_MG01.duration_p95    237
6      trace.to_MG02.duration_p95    231
7     trace.to_MG02.duration_mean    231
8         trace.to_MG02.row_count    231
9       trace.from_MG01.row_count    121
10  trace.from_MG01.duration_mean    121
11   trace.from_MG01.duration_p95    121
12   trace.from_IG02.duration_p95    121
13      trace.from_IG02.row_count    121
14  trace.from_IG02.duration_mean    121
15      trace.from_IG01.row_count    120
16   trace.from_IG01.duration_p95    120
17  trace.from_IG01.duration_mean    120
18   trace.from_MG02.duration_p95    120
19  trace.from_MG02.duration_mean    120,    total_rows  unique_cmdb_count                                 unique_cmdb_values             min_timestamp             max_timestamp
0         372                  6  Tomcat01,Tomcat02,Tomcat03,Tomcat04,apache01,a... 2021-03-23 01:00:00+08:00 2021-03-23 01:30:00+08:00,            log_name  count
0   log.error_count    182
1     log.row_count    182
2  log.gc_pause_max      8,    total_rows  unique_cmdb_count unique_cmdb_values min_timestamp max_timestamp
0           0                  0                              NaT           NaT, Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_metric (assumed loaded in kernel). Ensure ts column exists and is timezone-aware UTC+8.
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz and dtype
    df_metric['ts'] = pd.to_datetime(df_metric['ts']).dt.tz_convert(tz)

# 1) Compute global P95 for each (cmdb_id, kpi_name) using the full series (global threshold)
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(total_points='count')
)
# compute p95 separately to ensure correct alignment and name
p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)
metric_agg = metric_agg.merge(p95, on=['cmdb_id', 'kpi_name'])

# reorder columns as requested
metric_agg = metric_agg[['cmdb_id', 'kpi_name', 'global_p95', 'total_points']]

# 2) Filter original metric DataFrame to the incident window (inclusive)
start_ts = pd.Timestamp('2021-03-23 01:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2021-03-23 01:30:00').tz_localize(tz)
metric_window = df_metric[(df_metric['ts'] >= start_ts) & (df_metric['ts'] <= end_ts)].copy()

# 3) For each (cmdb_id, kpi_name) in metric_window compute required anomaly stats using global_p95 from metric_agg
# Merge window rows with global_p95
metric_window_with_p95 = metric_window.merge(metric_agg, on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomalies where value >= global_p95 (note: global_p95 computed from full series)
metric_window_with_p95['is_anomaly'] = metric_window_with_p95['value'] >= metric_window_with_p95['global_p95']

# Aggregate per group
agg_funcs = {
    'value': ['count', 'max'],
    'is_anomaly': 'sum',
    'ts': lambda s: s[ s.index.isin(s.index) ]  # placeholder; we'll compute earliest separately
}
grouped = metric_window_with_p95.groupby(['cmdb_id', 'kpi_name'], as_index=False)

# points_in_window and max_value_in_window and count_anomalies
metric_window_stats = (
    grouped
    .agg(points_in_window=('value', 'count'),
         max_value_in_window=('value', 'max'),
         count_anomalies=('is_anomaly', 'sum'))
)

# earliest_anomaly_timestamp: get min ts where is_anomaly True, else NaT
anomaly_earliest = (
    metric_window_with_p95[metric_window_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts': 'earliest_anomaly_timestamp'})
)

# Join stats with earliest timestamps and global_p95 from metric_agg
metric_anomalies = (
    metric_window_stats
    .merge(metric_agg, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomaly_earliest, on=['cmdb_id', 'kpi_name'], how='left')
)

# Reorder columns as requested
metric_anomalies = metric_anomalies[[
    'cmdb_id', 'kpi_name', 'global_p95', 'points_in_window',
    'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window'
]]

# Ensure proper dtypes: counts as int
metric_anomalies['points_in_window'] = metric_anomalies['points_in_window'].astype(int)
metric_anomalies['count_anomalies'] = metric_anomalies['count_anomalies'].astype(int)

# 4) Top 20 rows sorted by count_anomalies desc then earliest_anomaly_timestamp asc
metric_anomalies_sorted = metric_anomalies.sort_values(
    by=['count_anomalies', 'earliest_anomaly_timestamp'],
    ascending=[False, True]
).reset_index(drop=True)

metric_anomalies_top20 = metric_anomalies_sorted.head(20)

# sizes summary (compact)
sizes_df = pd.DataFrame({
    'table': ['metric_agg', 'metric_window', 'metric_anomalies'],
    'rows': [metric_agg.shape[0], metric_window.shape[0], metric_anomalies.shape[0]]
})

# Final compact outputs: top 20 anomalies table (selected columns) and sizes summary
metric_anomalies_top20, sizes_df

```
Out[2]:
```
summary = (
    "Summary of metric analysis:\n"
    "- metric_agg: computed global P95 thresholds for 1,745 (cmdb_id, kpi_name) groups.\n"
    "- metric_window: filtered metrics in 2021-03-23 01:00:00+08 to 2021-03-23 01:30:00+08 (inclusive) -> 30,676 rows.\n"
    "- metric_anomalies: joined window data with global P95 -> 1,745 groups evaluated (same count as metric_agg).\n"
    "- Top findings (from top-20): many groups have 31/31 points_in_window flagged as anomalies (value >= global_p95),\n"
    "  with earliest anomaly timestamp = 2021-03-23 01:00:00+08. Examples include JVM memory metrics for IG01/IG02,\n"
    "  JVM and OSLinux metrics for MG01/MG02, and several app.* metrics for ServiceTest2/5/7.\n"
    "- Interpretation: For these top groups, every sample in the 31-minute incident window exceeded their historical\n"
    "  95th-percentile, indicating sustained, system-wide elevated metric values starting at the window start.\n"
    "- Next steps: prioritize investigation of the listed components (IG01/IG02/MG01/MG02 and the ServiceTest services)\n"
    "  focusing on JVM memory and OS-level metrics as possible root causes.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name    global_p95  points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window
0           IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                31               31  2021-03-23 01:00:00+08:00         2.040136e+09
1           IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.805683e+07                31               31  2021-03-23 01:00:00+08:00         5.805683e+07
2           IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                31               31  2021-03-23 01:00:00+08:00         2.040136e+09
3           IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.811566e+07                31               31  2021-03-23 01:00:00+08:00         5.811566e+07
4           MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                31               31  2021-03-23 01:00:00+08:00         2.040136e+09
5           MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.894522e+07                31               31  2021-03-23 01:00:00+08:00         5.894522e+07
6           MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00                31               31  2021-03-23 01:00:00+08:00         0.000000e+00
7           MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2.000000e+00                31               31  2021-03-23 01:00:00+08:00         2.000000e+00
8           MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01                31               31  2021-03-23 01:00:00+08:00         4.000000e-01
9           MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00                31               31  2021-03-23 01:00:00+08:00         3.000000e+00
10          MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                31               31  2021-03-23 01:00:00+08:00         2.040136e+09
11          MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00                31               31  2021-03-23 01:00:00+08:00         0.000000e+00
12          MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00                31               31  2021-03-23 01:00:00+08:00         1.000000e+00
13          MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00                31               31  2021-03-23 01:00:00+08:00         1.000000e+00
14          MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00                31               31  2021-03-23 01:00:00+08:00         3.000000e+00
15  ServiceTest2                                             app.rr  1.000000e+02                31               31  2021-03-23 01:00:00+08:00         1.000000e+02
16  ServiceTest2                                             app.sr  1.000000e+02                31               31  2021-03-23 01:00:00+08:00         1.000000e+02
17  ServiceTest5                                             app.rr  1.000000e+02                31               31  2021-03-23 01:00:00+08:00         1.000000e+02
18  ServiceTest5                                             app.sr  1.000000e+02                31               31  2021-03-23 01:00:00+08:00         1.000000e+02
19  ServiceTest7                                             app.rr  1.000000e+02                31               31  2021-03-23 01:00:00+08:00         1.000000e+02,               table   rows
0        metric_agg   1745
1     metric_window  30676
2  metric_anomalies   1745)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_anomalies and metric_window from kernel
# Ensure metric_anomalies exists
# Compute severity_ratio with special handling for global_p95 == 0
ma = metric_anomalies.copy()

# severity_ratio logic:
# if global_p95 == 0:
#   if max_value_in_window == 0 -> NaN
#   else -> inf
# else severity_ratio = max_value_in_window / global_p95
ma['severity_ratio'] = np.where(
    ma['global_p95'] == 0,
    np.where(ma['max_value_in_window'] == 0, np.nan, np.inf),
    ma['max_value_in_window'] / ma['global_p95']
)

# consecutive_flag: True if count_anomalies == points_in_window AND count_anomalies >= 3
ma['consecutive_flag'] = (ma['count_anomalies'] == ma['points_in_window']) & (ma['count_anomalies'] >= 3)

# Filter to allowed root-cause component set
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
ma_filtered = ma[ma['cmdb_id'].isin(allowed_components)].copy()

# Count rows after filtering
filtered_count = ma_filtered.shape[0]

# Sort by severity_ratio desc (NaN last, inf highest), then count_anomalies desc
ma_filtered_sorted = ma_filtered.sort_values(
    by=['severity_ratio', 'count_anomalies'],
    ascending=[False, False]
).reset_index(drop=True)

# Select requested columns and top 20
ma_top20 = ma_filtered_sorted[[
    'cmdb_id', 'kpi_name', 'global_p95', 'points_in_window', 'count_anomalies',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio', 'consecutive_flag'
]].head(20)

# Return compact outputs: top-20 table and the filtered row count
ma_top20, filtered_count

```
Out[3]:
```
summary = (
    "Summary of severity and consecutive-anomaly analysis (allowed components only):\n\n"
    "- Rows after filtering to allowed components: 1,701 groups.\n\n"
    "- Top findings (top-20 by severity_ratio then count):\n"
    "  * Several groups have severity_ratio = inf because their global P95 = 0 while max_value_in_window > 0.\n"
    "    Examples: MG02 (OSLinux network TCP-FIN-WAIT), Tomcat01 and Tomcat02 (http-nio thread metrics),\n"
    "    Mysql01 (transaction lock metrics). These entries also have consecutive_flag = True (all points in the\n"
    "    30–31 minute window were anomalies), indicating sustained non-zero activity against a historical P95 of 0.\n"
    "  * Other top entries show elevated severity_ratio in the ~3–6 range (e.g., CPU WIO, disk I/O metrics on Tomcat/MG),\n"
    "    but most of these are not consecutive across the entire window (consecutive_flag = False).\n\n"
    "- Interpretation and priorities:\n"
    "  1) Highest priority: groups with severity_ratio = inf and consecutive_flag = True — sustained anomalous signals\n"
    "     where historical baseline P95 was zero. Investigate MG02, Tomcat01/02, and Mysql01 transaction-lock metrics first.\n"
    "  2) Second priority: groups with large numeric severity_ratio (>3) and multiple anomaly points — investigate disk I/O and\n"
    "     CPU WIO metrics on Tomcat and MG nodes.\n\n"
    "- Actionable next steps: examine process/thread dumps and JVM/OS metrics on the prioritized hosts, check MySQL locks,\n"
    "  and correlate with logs/traces to find root causes for sustained anomalies.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_p95  points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio  consecutive_flag
0       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT      0.000000                31               31  2021-03-23 01:00:00+08:00             1.000000             inf              True
1   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      0.000000                31               31  2021-03-23 01:00:00+08:00             1.000000             inf              True
2   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      0.000000                31               31  2021-03-23 01:00:00+08:00             1.000000             inf              True
3    Mysql01               Mysql-MySQL_3306_Max trx rows locked      0.000000                30               30  2021-03-23 01:01:00+08:00             1.000000             inf              True
4    Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified      0.000000                30               30  2021-03-23 01:01:00+08:00             1.000000             inf              True
5    Mysql01         Mysql-MySQL_3306_max trx lock memory bytes      0.000000                30               30  2021-03-23 01:01:00+08:00          1136.000000             inf              True
6       MG02                             OSLinux-CPU_CPU_CPUWio      0.792577                30                2  2021-03-23 01:01:00+08:00             5.212187        6.576250             False
7   Tomcat02                             OSLinux-CPU_CPU_CPUWio      3.255701                30                2  2021-03-23 01:00:00+08:00            13.313105        4.089167             False
8   Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite   1916.833325                16                1  2021-03-23 01:02:00+08:00          7565.733300        3.946996             False
9   Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...      2.850000                16                1  2021-03-23 01:00:00+08:00            10.500000        3.684211             False
10  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...      2.550000                16                1  2021-03-23 01:00:00+08:00             9.300000        3.647059             False
11  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle     19.671698                16                1  2021-03-23 01:08:00+08:00            68.517300        3.483039             False
12  Tomcat03                             OSLinux-CPU_CPU_CPUWio      3.220115                30                2  2021-03-23 01:00:00+08:00            10.895900        3.383699             False
13      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead     29.319990                15                1  2021-03-23 01:01:00+08:00            97.733300        3.333333             False
14  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps      0.040000                15                1  2021-03-23 01:17:00+08:00             0.133333        3.333333             False
15  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps     19.635000                15                1  2021-03-23 01:01:00+08:00            65.450000        3.333333             False
16  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead  17191.640010                15                1  2021-03-23 01:01:00+08:00         57305.466700        3.333333             False
17      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps      0.054990                15                1  2021-03-23 01:01:00+08:00             0.183300        3.333333             False
18      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps      1.200000                15                1  2021-03-23 01:01:00+08:00             4.000000        3.333333             False
19  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead      0.720000                15                1  2021-03-23 01:17:00+08:00             2.400000        3.333333             False, 1701)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# ensure timezone variable
tz = pytz.timezone('Asia/Shanghai')

# Ensure df_trace and ts exist and ts is timezone-aware UTC+8
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    df_trace['ts'] = pd.to_datetime(df_trace['ts']).dt.tz_convert(tz)

# 1) Compute global P95 and total_points for each (cmdb_id, trace_name)
trace_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)
trace_agg_counts = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(total_points='count')
)
trace_agg = trace_agg_counts.merge(trace_p95, on=['cmdb_id', 'trace_name'])
trace_agg = trace_agg[['cmdb_id', 'trace_name', 'global_p95', 'total_points']]

# 2) Filter trace.csv to the incident window (inclusive)
start_ts = pd.Timestamp('2021-03-23 01:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2021-03-23 01:30:00').tz_localize(tz)
trace_window = df_trace[(df_trace['ts'] >= start_ts) & (df_trace['ts'] <= end_ts)].copy()

# 3) For each group in trace_window compute points_in_window, count_anomalies, earliest_anomaly_timestamp, max_value_in_window
# Merge window with global_p95
trace_window_with_p95 = trace_window.merge(trace_agg, on=['cmdb_id', 'trace_name'], how='left')

# Flag anomalies (value >= global_p95). If global_p95 is NaN, treat as False
trace_window_with_p95['is_anomaly'] = trace_window_with_p95['value'] >= trace_window_with_p95['global_p95']

# Aggregate stats
trace_stats = (
    trace_window_with_p95
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(points_in_window=('value', 'count'),
         count_anomalies=('is_anomaly', 'sum'),
         max_value_in_window=('value', 'max'))
)

# earliest anomaly timestamp per group (min ts where is_anomaly True)
earliest_anom = (
    trace_window_with_p95[trace_window_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts': 'earliest_anomaly_timestamp'})
)

# Join into trace_anomalies
trace_anomalies = (
    trace_stats
    .merge(trace_agg, on=['cmdb_id', 'trace_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')
)

# Reorder columns
trace_anomalies = trace_anomalies[[
    'cmdb_id', 'trace_name', 'global_p95', 'points_in_window',
    'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window'
]]

# Ensure integer counts
trace_anomalies['points_in_window'] = trace_anomalies['points_in_window'].astype(int)
trace_anomalies['count_anomalies'] = trace_anomalies['count_anomalies'].astype(int)

# 4) Add severity_ratio and consecutive_flag
trace_anomalies['severity_ratio'] = np.where(
    trace_anomalies['global_p95'] == 0,
    np.where(trace_anomalies['max_value_in_window'] == 0, np.nan, np.inf),
    trace_anomalies['max_value_in_window'] / trace_anomalies['global_p95']
)
trace_anomalies['consecutive_flag'] = (
    (trace_anomalies['count_anomalies'] == trace_anomalies['points_in_window']) &
    (trace_anomalies['count_anomalies'] >= 3)
)

# 5) Filter to allowed root-cause component set
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
trace_anomalies_filtered = trace_anomalies[trace_anomalies['cmdb_id'].isin(allowed_components)].copy()

# 6) Prepare compact output: top 20 rows sorted by count_anomalies desc then severity_ratio desc
trace_anomalies_sorted = trace_anomalies_filtered.sort_values(
    by=['count_anomalies', 'severity_ratio'],
    ascending=[False, False]
).reset_index(drop=True)

trace_top20 = trace_anomalies_sorted[[
    'cmdb_id', 'trace_name', 'global_p95', 'points_in_window',
    'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window',
    'severity_ratio', 'consecutive_flag'
]].head(20)

# Sizes summary
sizes = pd.DataFrame({
    'table': ['trace_agg', 'trace_window', 'trace_anomalies_filtered'],
    'rows': [trace_agg.shape[0], trace_window.shape[0], trace_anomalies_filtered.shape[0]]
})

# Return compact results
trace_top20, sizes

```
Out[4]:
```
summary = (
    "Trace analysis summary (allowed components):\n"
    "- trace_agg: 231 (cmdb_id, trace_name) groups computed with global P95.\n"
    "- trace_window: 6,798 trace records in 2021-03-23 01:00:00+08 to 01:30:00+08.\n"
    "- trace_anomalies (filtered to allowed components): 171 groups.\n\n"
    "Top observations (from top-20):\n"
    "- Highest count_anomalies = 4: MG01.trace.from_Tomcat04.row_count and Tomcat04.trace.to_MG01.row_count\n"
    "  (points_in_window=30, global_p95=29, max=31, severity_ratio≈1.07, earliest anomaly 01:01:00+08).\n"
    "- Several groups have count_anomalies = 3 (e.g., MG02 <-> Tomcat1/3/2 row_count flows) with severity_ratio ~1.04–1.85.\n"
    "- Some duration_p95 traces show high severity_ratio (e.g., >2 and up to ~4) but with fewer anomaly points (count 2),\n"
    "  indicating short but strong latency spikes (examples on dockerA/B and MG nodes).\n"
    "- No top-20 group has consecutive_flag = True (no group had anomalies at every sample in-window with >=3 points).\n\n"
    "Interpretation and next steps:\n"
    "1) Prioritize MG01 <-> Tomcat04 row_count path (highest sustained anomaly count) for investigation of request routing\n"
    "   and queuing issues.\n"
    "2) Investigate MG02 <-> Tomcat interactions where row_count and duration_p95 are elevated (count 3 anomalies).\n"
    "3) For high severity_ratio but short spikes (duration_p95 entries), correlate with traces/logs to find transient latency causes.\n"
    "4) Use these prioritized trace groups together with metric anomalies (JVM/OS/disk) and logs to locate root cause.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  global_p95  points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio  consecutive_flag
0       MG01      trace.from_Tomcat04.row_count   29.000000                30                4  2021-03-23 01:01:00+08:00            31.000000        1.068966             False
1   Tomcat04            trace.to_MG01.row_count   29.000000                30                4  2021-03-23 01:01:00+08:00            31.000000        1.068966             False
2       MG02   trace.from_Tomcat03.duration_p95    0.230500                30                3  2021-03-23 01:08:00+08:00             0.425300        1.845119             False
3   Tomcat03         trace.to_MG02.duration_p95    0.230500                30                3  2021-03-23 01:08:00+08:00             0.425300        1.845119             False
4       MG02      trace.from_Tomcat01.row_count   26.000000                30                3  2021-03-23 01:06:00+08:00            38.000000        1.461538             False
5   Tomcat01            trace.to_MG02.row_count   26.000000                30                3  2021-03-23 01:06:00+08:00            38.000000        1.461538             False
6       MG02      trace.from_dockerA1.row_count   10.000000                27                3  2021-03-23 01:03:00+08:00            13.000000        1.300000             False
7       MG02        trace.to_dockerA1.row_count   21.000000                30                3  2021-03-23 01:03:00+08:00            24.000000        1.142857             False
8       MG01        trace.to_dockerB2.row_count   22.000000                31                3  2021-03-23 01:01:00+08:00            25.000000        1.136364             False
9       IG02        trace.to_Tomcat02.row_count   29.000000                30                3  2021-03-23 01:00:00+08:00            30.000000        1.034483             False
10  Tomcat02          trace.from_IG02.row_count   29.000000                30                3  2021-03-23 01:00:00+08:00            30.000000        1.034483             False
11      MG02   trace.from_dockerB1.duration_p95    0.061987                28                2  2021-03-23 01:12:00+08:00             0.251800        4.062109             False
12      MG01     trace.to_dockerA2.duration_p95    0.138920                30                2  2021-03-23 01:26:00+08:00             0.433400        3.119781             False
13      MG02   trace.from_dockerA1.duration_p95    0.014730                27                2  2021-03-23 01:01:00+08:00             0.044000        2.987101             False
14      MG01  trace.from_Tomcat04.duration_mean    0.056538                30                2  2021-03-23 01:13:00+08:00             0.167333        2.959629             False
15  Tomcat04        trace.to_MG01.duration_mean    0.056538                30                2  2021-03-23 01:13:00+08:00             0.167333        2.959629             False
16      MG02   trace.from_dockerA2.duration_p95    0.243440                25                2  2021-03-23 01:26:00+08:00             0.663300        2.724696             False
17      MG01   trace.from_dockerA2.duration_p95    0.143690                27                2  2021-03-23 01:01:00+08:00             0.356000        2.477556             False
18      MG01   trace.from_Tomcat04.duration_p95    0.300062                30                2  2021-03-23 01:13:00+08:00             0.694500        2.314518             False
19  Tomcat04         trace.to_MG01.duration_p95    0.300062                30                2  2021-03-23 01:13:00+08:00             0.694500        2.314518             False,                       table  rows
0                 trace_agg   231
1              trace_window  6798
2  trace_anomalies_filtered   171)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_log is loaded and has timezone-aware 'ts'
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    df_log['ts'] = pd.to_datetime(df_log['ts']).dt.tz_convert(tz)

# 1) Compute global P95 and total_points for each (cmdb_id, log_name)
log_p95 = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)
log_counts = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(total_points='count')
)
log_agg = log_counts.merge(log_p95, on=['cmdb_id', 'log_name'])
log_agg = log_agg[['cmdb_id', 'log_name', 'global_p95', 'total_points']]

# 2) Filter to incident window (inclusive)
start_ts = pd.Timestamp('2021-03-23 01:00:00').tz_localize(tz)
end_ts = pd.Timestamp('2021-03-23 01:30:00').tz_localize(tz)
log_window = df_log[(df_log['ts'] >= start_ts) & (df_log['ts'] <= end_ts)].copy()

# 3) For each (cmdb_id, log_name) in window compute stats using global_p95
log_win_with_p95 = log_window.merge(log_agg, on=['cmdb_id', 'log_name'], how='left')

# Flag anomalies (treat NaN global_p95 as no anomaly)
log_win_with_p95['is_anomaly'] = False
mask_has_p95 = log_win_with_p95['global_p95'].notna()
log_win_with_p95.loc[mask_has_p95, 'is_anomaly'] = (
    log_win_with_p95.loc[mask_has_p95, 'value'] >= log_win_with_p95.loc[mask_has_p95, 'global_p95']
)

# Aggregate stats
log_stats = (
    log_win_with_p95
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(points_in_window=('value', 'count'),
         count_anomalies=('is_anomaly', 'sum'),
         max_value_in_window=('value', 'max'))
)

# earliest anomaly timestamp
earliest_anom = (
    log_win_with_p95[log_win_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'log_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts': 'earliest_anomaly_timestamp'})
)

# Join into log_anomalies
log_anomalies = (
    log_stats
    .merge(log_agg, on=['cmdb_id', 'log_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'log_name'], how='left')
)

# Reorder columns
log_anomalies = log_anomalies[[
    'cmdb_id', 'log_name', 'global_p95', 'points_in_window',
    'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window'
]]

# Ensure integer counts
log_anomalies['points_in_window'] = log_anomalies['points_in_window'].astype(int)
log_anomalies['count_anomalies'] = log_anomalies['count_anomalies'].astype(int)

# 4) Add severity_ratio and consecutive_flag
log_anomalies['severity_ratio'] = np.where(
    log_anomalies['global_p95'] == 0,
    np.where(log_anomalies['max_value_in_window'] == 0, np.nan, np.inf),
    log_anomalies['max_value_in_window'] / log_anomalies['global_p95']
)
log_anomalies['consecutive_flag'] = (
    (log_anomalies['count_anomalies'] == log_anomalies['points_in_window']) &
    (log_anomalies['count_anomalies'] >= 3)
)

# 5) Filter to allowed root-cause components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
log_anomalies_filtered = log_anomalies[log_anomalies['cmdb_id'].isin(allowed_components)].copy()

# 6) Prepare compact output: top 20 sorted by count_anomalies desc then severity_ratio desc
log_anomalies_sorted = log_anomalies_filtered.sort_values(
    by=['count_anomalies', 'severity_ratio'],
    ascending=[False, False]
).reset_index(drop=True)

log_top20 = log_anomalies_sorted[[
    'cmdb_id', 'log_name', 'global_p95', 'points_in_window',
    'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window',
    'severity_ratio', 'consecutive_flag'
]].head(20)

# Sizes summary
sizes = pd.DataFrame({
    'table': ['log_agg', 'log_window', 'log_anomalies_filtered'],
    'rows': [log_agg.shape[0], log_window.shape[0], log_anomalies_filtered.shape[0]]
})

# Return compact results
log_top20, sizes

```
Out[5]:
```
Summary of log analysis (allowed components):

- Counts:
  - log_agg: 16 (cmdb_id, log_name) groups with global P95 computed.
  - log_window: 372 log records in 2021-03-23 01:00:00+08 to 01:30:00+08.
  - log_anomalies (after filtering to allowed components): 16 groups.

- Top findings (top-20 shown, key highlights):
  - Several entries for log.error_count (Tomcat01, apache02, Tomcat02, Tomcat03, Tomcat04, apache01) have global_p95 = 0, points_in_window = 30–31, count_anomalies = same as points_in_window, earliest timestamp = 01:00:00+08, max_value_in_window = 0, severity_ratio = NaN, consecutive_flag = True.
    - Note: these are flagged because value >= global_p95 and global_p95==0; since values are 0 across the window, they are not actionable errors but an artifact of thresholding.
  - log.row_count spikes:
    - Tomcat03.log.row_count: global_p95=50, points_in_window=30, count_anomalies=3, max=51, earliest=01:03 -> severity_ratio ≈1.02.
    - Tomcat01/Tomcat04/Tomcat02/apache01/apache02 have smaller row_count increases (count_anomalies 2 each, severity_ratio ~1.10–1.23).
  - log.gc_pause_max: a few small GC pause increases (Tomcat04: 2 of 3 points flagged; Tomcat02/01/03 single occurrences) with severity_ratio ≈1.0–1.02.

- Interpretation and recommendations:
  1) Ignore the log.error_count rows that are flagged purely because both historical P95 and current values are zero (these are not meaningful anomalies). This explains multiple entries with consecutive_flag=True but max_value=0.
  2) Prioritize investigating elevated log.row_count on Tomcat nodes (especially Tomcat03 with 3 anomaly points and Tomcat01/Tomcat04 with 2 points). These indicate increased request/row activity during the incident window.
  3) Check GC pause increases on Tomcat04 (and single points on other Tomcats) as secondary signals to correlate with CPU/JVM/trace anomalies.
  4) Correlate the above with metric and trace findings (JVM memory, CPU WIO, and MG<->Tomcat trace row_count anomalies) to locate root cause.

If you want, I can produce the filtered table rows (compact) for the prioritized log groups for quick inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  global_p95  points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio  consecutive_flag
0   Tomcat01   log.error_count      0.0000                31               31  2021-03-23 01:00:00+08:00                 0.00             NaN              True
1   apache02   log.error_count      0.0000                31               31  2021-03-23 01:00:00+08:00                 0.00             NaN              True
2   Tomcat02   log.error_count      0.0000                30               30  2021-03-23 01:00:00+08:00                 0.00             NaN              True
3   Tomcat03   log.error_count      0.0000                30               30  2021-03-23 01:00:00+08:00                 0.00             NaN              True
4   Tomcat04   log.error_count      0.0000                30               30  2021-03-23 01:00:00+08:00                 0.00             NaN              True
5   apache01   log.error_count      0.0000                30               30  2021-03-23 01:00:00+08:00                 0.00             NaN              True
6   Tomcat03     log.row_count     50.0000                30                3  2021-03-23 01:03:00+08:00                51.00        1.020000             False
7   Tomcat01     log.row_count     54.5000                31                2  2021-03-23 01:07:00+08:00                67.00        1.229358             False
8   Tomcat04     log.row_count     49.5500                30                2  2021-03-23 01:07:00+08:00                59.00        1.190716             False
9   Tomcat02     log.row_count     52.1000                30                2  2021-03-23 01:01:00+08:00                58.00        1.113244             False
10  apache01     log.row_count     99.4000                30                2  2021-03-23 01:07:00+08:00               110.00        1.106640             False
11  apache02     log.row_count    101.5000                31                2  2021-03-23 01:07:00+08:00               112.00        1.103448             False
12  Tomcat04  log.gc_pause_max      0.0600                 3                2  2021-03-23 01:15:00+08:00                 0.06        1.000000             False
13  Tomcat02  log.gc_pause_max      0.0590                 2                1  2021-03-23 01:04:00+08:00                 0.06        1.016949             False
14  Tomcat01  log.gc_pause_max      0.0895                 2                1  2021-03-23 01:06:00+08:00                 0.09        1.005587             False
15  Tomcat03  log.gc_pause_max      0.0700                 1                1  2021-03-23 01:09:00+08:00                 0.07        1.000000             False,                     table  rows
0                 log_agg    16
1              log_window   372
2  log_anomalies_filtered    16)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required kernel variables exist
# metric_anomalies should be present from prior steps
# trace_anomalies_filtered may exist; prefer it, else fall back to trace_anomalies
if 'metric_anomalies' not in globals():
    raise RuntimeError("metric_anomalies not found in kernel.")
ta = trace_anomalies_filtered if 'trace_anomalies_filtered' in globals() else (trace_anomalies if 'trace_anomalies' in globals() else None)
if ta is None:
    raise RuntimeError("trace_anomalies_filtered or trace_anomalies not found in kernel.")

components = ["MG01", "Tomcat04"]

# Work on a copy to avoid mutating kernel originals
ma = metric_anomalies.copy()

# Ensure datetime dtype for earliest_anomaly_timestamp
if 'earliest_anomaly_timestamp' in ma.columns:
    ma['earliest_anomaly_timestamp'] = pd.to_datetime(ma['earliest_anomaly_timestamp'])

# Add severity_ratio if missing
if 'severity_ratio' not in ma.columns:
    ma['severity_ratio'] = np.where(
        ma['global_p95'] == 0,
        np.where(ma['max_value_in_window'] == 0, np.nan, np.inf),
        ma['max_value_in_window'] / ma['global_p95']
    )

# Add consecutive_flag if missing
if 'consecutive_flag' not in ma.columns:
    ma['consecutive_flag'] = (ma['count_anomalies'] == ma['points_in_window']) & (ma['count_anomalies'] >= 3)

# 1) Filter metric_anomalies for MG01 and Tomcat04 and sort by consecutive_flag (True first) then severity_ratio desc
metric_evidence = (
    ma[ma['cmdb_id'].isin(components)]
    .copy()
)
metric_evidence = metric_evidence.sort_values(
    by=['consecutive_flag', 'severity_ratio'],
    ascending=[False, False]
)[[
    'cmdb_id', 'kpi_name', 'global_p95', 'points_in_window', 'count_anomalies',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio', 'consecutive_flag'
]].reset_index(drop=True).head(20)

# 2) Per-component earliest metric anomaly time and number of kpis with consecutive_flag=True
metric_earliest = (
    ma[ma['cmdb_id'].isin(components)]
    .groupby('cmdb_id', as_index=False)
    .agg(
        earliest_metric_anomaly_timestamp=('earliest_anomaly_timestamp', 'min'),
        num_faulty_kpis=('consecutive_flag', 'sum')
    )
)
metric_earliest['num_faulty_kpis'] = metric_earliest['num_faulty_kpis'].astype(int)

# 3) From trace_anomalies (filtered), select rows involving MG01 and Tomcat04 by name patterns
tb = ta.copy()
# ensure datetime dtype
if 'earliest_anomaly_timestamp' in tb.columns:
    tb['earliest_anomaly_timestamp'] = pd.to_datetime(tb['earliest_anomaly_timestamp'])

pattern_keywords = ['Tomcat04', 'MG01', 'to_MG01', 'from_Tomcat04']
mask_comp = tb['cmdb_id'].isin(components)
mask_name = tb['trace_name'].astype(str).apply(lambda s: any(k in s for k in pattern_keywords))
trace_evidence = tb[mask_comp & mask_name].copy()

# Ensure severity_ratio exists; compute if missing
if 'severity_ratio' not in trace_evidence.columns:
    trace_evidence['severity_ratio'] = np.where(
        trace_evidence['global_p95'] == 0,
        np.where(trace_evidence['max_value_in_window'] == 0, np.nan, np.inf),
        trace_evidence['max_value_in_window'] / trace_evidence['global_p95']
    )

trace_evidence = trace_evidence.sort_values(
    by=['count_anomalies', 'severity_ratio'],
    ascending=[False, False]
)[[
    'cmdb_id', 'trace_name', 'global_p95', 'points_in_window', 'count_anomalies',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
]].reset_index(drop=True).head(20)

# 4) Earliest trace anomaly timestamp among these returned trace rows per cmdb_id
trace_earliest = (
    trace_evidence.groupby('cmdb_id', as_index=False)
    .agg(earliest_trace_anomaly_timestamp=('earliest_anomaly_timestamp', 'min'))
)

# Counts: number of metric_anomalies rows for MG01 and Tomcat04, and number of matching trace_anomalies rows for MG01 and Tomcat04
metric_counts = pd.DataFrame({
    'cmdb_id': components,
    'metric_anomalies_rows': [int(ma[ma['cmdb_id'] == c].shape[0]) for c in components]
})

trace_counts = pd.DataFrame({
    'cmdb_id': components,
    'matching_trace_anomalies_rows': [int(tb[(tb['cmdb_id'] == c) & (tb['trace_name'].astype(str).apply(lambda s: any(k in s for k in pattern_keywords)))].shape[0]) for c in components]
})

# Return compact tables (limited to top 20 where applicable)
metric_evidence, metric_earliest, trace_evidence, trace_earliest, metric_counts, trace_counts

```
Out[6]:
```
summary = (
    "Compact evidence summary to choose between MG01 and Tomcat04 and identify root-fault time:\n\n"
    "Key facts from the analysis:\n"
    "- Earliest metric anomalies: both MG01 and Tomcat04 show their first metric anomaly at 2021-03-23 01:00:00+08.\n"
    "- Number of kpis with consecutive (persistent) faults: Tomcat04 = 64, MG01 = 46 (Tomcat04 has more faulty KPIs).\n"
    "- Total metric_anomalies rows: Tomcat04 = 122, MG01 = 104 (Tomcat04 has more affected metric groups overall).\n"
    "- Trace anomalies involving the pair: MG01.trace.from_Tomcat04.row_count and Tomcat04.trace.to_MG01.row_count\n"
    "  both show earliest trace anomaly at 2021-03-23 01:01:00+08, with 4 anomalous points each (matching interactions).\n"
    "- Matching trace_anomalies rows found for each: MG01 = 3, Tomcat04 = 3 (same count of relevant trace groups).\n\n"
    "Interpretation and conclusion:\n"
    "- The incident root-fault time is approximately 2021-03-23 01:00:00+08 (metric anomalies begin then); related\n"
    "  request/row-count trace anomalies appear by ~01:01:00+08, indicating propagation within a minute.\n"
    "- Both hosts are implicated and interact: traces show MG01 <-> Tomcat04 row_count anomalies at ~01:01.\n"
    "- Tomcat04 shows stronger metric evidence of being the primary affected host (more consecutive faulty KPIs\n"
    "  and more total anomalous metric groups). Therefore Tomcat04 is the higher-priority candidate for the root\n"
    "  of the fault, with MG01 likely involved downstream or as an immediately affected peer.\n\n"
    "Recommended next steps (investigate in this order):\n"
    "1) Inspect Tomcat04: JVM memory usage, thread pools (http-nio), GC pauses, and application logs around 01:00:00+08.\n"
    "2) Correlate Tomcat04 findings with traces showing elevated row_count to/from MG01 at 01:01:00+08.\n"
    "3) Check MG01 filesystem/IO and network metrics (many filesystem metrics are persistently anomalous at ~01:01) to\n"
    "   determine whether MG01 is a downstream victim or a contributor.\n\n"
    "Bottom line: root-fault time ≈ 2021-03-23 01:00:00+08. Prioritize Tomcat04 for root cause investigation while\n"
    "simultaneously examining MG01 due to confirmed trace-level interaction at ~01:01:00+08."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_p95  points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio  consecutive_flag
0       MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01                31               31  2021-03-23 01:00:00+08:00         4.000000e-01        1.333333              True
1   Tomcat04       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.200000e+01                22               22  2021-03-23 01:00:00+08:00         9.800000e+01        1.065217              True
2       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                31               31  2021-03-23 01:00:00+08:00         2.040136e+09        1.000000              True
3       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.894522e+07                31               31  2021-03-23 01:00:00+08:00         5.894522e+07        1.000000              True
4       MG01  OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPer...  1.636648e+01                15               15  2021-03-23 01:01:00+08:00         1.636648e+01        1.000000              True
5       MG01     OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity  1.541695e+01                15               15  2021-03-23 01:01:00+08:00         1.541695e+01        1.000000              True
6       MG01  OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUse...  4.714970e-01                15               15  2021-03-23 01:01:00+08:00         4.714970e-01        1.000000              True
7       MG01  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCa...  4.268120e-01                15               15  2021-03-23 01:01:00+08:00         4.268120e-01        1.000000              True
8       MG01  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSIn...  4.196000e-03                15               15  2021-03-23 01:01:00+08:00         4.196000e-03        1.000000              True
9       MG01     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity  2.706176e+00                15               15  2021-03-23 01:01:00+08:00         2.706176e+00        1.000000              True
10      MG01  OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUse...  2.191160e-01                15               15  2021-03-23 01:01:00+08:00         2.191160e-01        1.000000              True
11      MG01  OSLinux-OSLinux_FILESYSTEM_----tmp_FSInodeUsed...  1.586900e-02                15               15  2021-03-23 01:01:00+08:00         1.586900e-02        1.000000              True
12      MG01      OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace  9.952000e+03                11               11  2021-03-23 01:01:00+08:00         9.952000e+03        1.000000              True
13      MG01    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent  1.636650e+01                 9                9  2021-03-23 01:01:00+08:00         1.636650e+01        1.000000              True
14      MG01           OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace  4.318000e+03                11               11  2021-03-23 01:01:00+08:00         4.318000e+03        1.000000              True
15      MG01  OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace  7.680000e+02                11               11  2021-03-23 01:01:00+08:00         7.680000e+02        1.000000              True
16      MG01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity  1.541700e+01                 9                9  2021-03-23 01:01:00+08:00         1.541700e+01        1.000000              True
17      MG01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...  4.715000e-01                 9                9  2021-03-23 01:01:00+08:00         4.715000e-01        1.000000              True
18      MG01       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace  1.400000e+02                11               11  2021-03-23 01:01:00+08:00         1.400000e+02        1.000000              True
19      MG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvail...  3.730000e+03                11               11  2021-03-23 01:01:00+08:00         3.730000e+03        1.000000              True,     cmdb_id earliest_metric_anomaly_timestamp  num_faulty_kpis
0      MG01         2021-03-23 01:00:00+08:00               46
1  Tomcat04         2021-03-23 01:00:00+08:00               64,     cmdb_id                         trace_name  global_p95  points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0      MG01      trace.from_Tomcat04.row_count   29.000000                30                4  2021-03-23 01:01:00+08:00            31.000000        1.068966
1  Tomcat04            trace.to_MG01.row_count   29.000000                30                4  2021-03-23 01:01:00+08:00            31.000000        1.068966
2      MG01  trace.from_Tomcat04.duration_mean    0.056538                30                2  2021-03-23 01:13:00+08:00             0.167333        2.959629
3  Tomcat04        trace.to_MG01.duration_mean    0.056538                30                2  2021-03-23 01:13:00+08:00             0.167333        2.959629
4      MG01   trace.from_Tomcat04.duration_p95    0.300062                30                2  2021-03-23 01:13:00+08:00             0.694500        2.314518
5  Tomcat04         trace.to_MG01.duration_p95    0.300062                30                2  2021-03-23 01:13:00+08:00             0.694500        2.314518,     cmdb_id earliest_trace_anomaly_timestamp
0      MG01        2021-03-23 01:01:00+08:00
1  Tomcat04        2021-03-23 01:01:00+08:00,     cmdb_id  metric_anomalies_rows
0      MG01                    104
1  Tomcat04                    122,     cmdb_id  matching_trace_anomalies_rows
0      MG01                              3
1  Tomcat04                              3)```
```